# 训练工程：日志、Checkpoint 与复现实验

## 本节目标

一个学习项目不仅要能训练，还要能复现实验、记录结果和保存模型。本节补充训练工程基础。

完成本节后，你应该能够：

- 说清本节主题解决了什么问题。
- 读懂核心 PyTorch API 的输入输出。
- 运行最小代码示例并解释结果。
- 修改实验参数并观察变化。

## 背景问题

本节关注的问题是：一个学习项目不仅要能训练，还要能复现实验、记录结果和保存模型。本节补充训练工程基础。

学习时建议先运行代码，再回头看解释。对初学者来说，能观察 shape、loss、accuracy 或可视化结果，比只记概念更重要。

## 核心概念

- 输入和输出的 shape 必须明确。
- loss 用来描述模型当前预测和目标之间的差距。
- 优化器根据梯度更新模型参数。
- 实验观察需要结合曲线、数值和样本可视化。
- 调试时优先检查 shape、dtype、学习率和训练/评估模式。

这里的关键不是堆公式，而是把概念落实到可运行代码。

## 最小代码示例：准备数据和模型

这个代码块用于观察 `准备数据和模型`。运行后请重点看输出 shape、loss 或图像结果是否符合预期。

In [ ]:
import torch
from torch import nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)
from pathlib import Path

x = torch.randn(300, 4)
y = (x[:, 0] + x[:, 1] * 0.5 - x[:, 2] > 0).long()
model = nn.Sequential(nn.Linear(4, 16), nn.ReLU(), nn.Linear(16, 2))
loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=0.01)
log = []

## 最小代码示例：记录训练日志

这个代码块用于观察 `记录训练日志`。运行后请重点看输出 shape、loss 或图像结果是否符合预期。

In [ ]:
for epoch in range(60):
    logits = model(x)
    loss = loss_fn(logits, y)
    opt.zero_grad()
    loss.backward()
    opt.step()
    with torch.no_grad():
        acc = (model(x).argmax(1) == y).float().mean().item()
    log.append({"epoch": epoch, "loss": loss.item(), "acc": acc})

print(log[-3:])

## 最小代码示例：保存和加载 checkpoint

这个代码块用于观察 `保存和加载 checkpoint`。运行后请重点看输出 shape、loss 或图像结果是否符合预期。

In [ ]:
checkpoint_dir = Path("..") / "results"
checkpoint_dir.mkdir(exist_ok=True)
checkpoint_path = checkpoint_dir / "demo_checkpoint.pt"

torch.save({
    "model_state": model.state_dict(),
    "optimizer_state": opt.state_dict(),
    "log": log,
}, checkpoint_path)

new_model = nn.Sequential(nn.Linear(4, 16), nn.ReLU(), nn.Linear(16, 2))
state = torch.load(checkpoint_path)
new_model.load_state_dict(state["model_state"])
print("loaded log length:", len(state["log"]))

## 完整实验：绘制日志

这个代码块用于观察 `绘制日志`。运行后请重点看输出 shape、loss 或图像结果是否符合预期。

In [ ]:
plt.figure(figsize=(8, 3))
plt.subplot(1, 2, 1)
plt.plot([row["loss"] for row in log])
plt.title("Loss")
plt.grid(alpha=0.3)
plt.subplot(1, 2, 2)
plt.plot([row["acc"] for row in log])
plt.title("Accuracy")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 实验观察

日志帮助复盘训练过程，checkpoint 支持恢复模型。开源项目中，清楚记录实验设置和结果比只给最终代码更有价值。

从运行结果可以看到，代码输出不只是“能跑”，还可以帮助判断模型是否在按预期学习。建议把观察写进实验记录，而不是只保留最终准确率。

## 常见错误

- shape 不匹配：先打印输入、输出和标签 shape。
- dtype 不正确：分类标签通常是 `torch.long`，回归目标通常是 float。
- 忘记 `optimizer.zero_grad()`：梯度会累积。
- 学习率不合理：过大震荡，过小收敛慢。
- 评估阶段忘记 `model.eval()` 或 `torch.no_grad()`。

调试时可以先缩小数据集，确认模型能否在小样本上过拟合。

## 概念问答

**Q：为什么要从最小代码示例开始？**  
A：最小示例能隔离核心概念，降低同时排查多个问题的难度。

**Q：为什么每个实验都要看曲线或可视化？**  
A：单个最终数值不一定能解释训练过程，曲线和样本结果更容易暴露问题。

**Q：如果代码能运行但结果不好，先查什么？**  
A：优先检查数据、shape、label dtype、loss 使用方式和学习率。

## 延伸练习

- 把日志保存成 CSV。
- 增加验证集并保存最佳 checkpoint。

这些练习不需要一次完成。建议每次只改一个变量，并记录改动前后的结果。

## 小结

本节把一个深度学习概念落到了可运行的 PyTorch 实验中。继续学习下一节前，建议确认自己能解释：输入是什么、模型做了什么、loss 如何计算、结果是否符合预期。